# V5 — Cost Contribution to Alpha

Decompose each region's baseline $\alpha_i$ into:
- **Cost-predicted component**: $\mu_\lambda \cdot c_i$
- **Residual component**: $\alpha_i - \mu_\lambda \cdot c_i$

And compute the ratio: how much of alpha is explained by budget?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
raw_df  = pd.read_csv('../../data/models/v5/raw_samples.csv')
regions = pd.read_csv('../../data/regions.csv')['region'].tolist()
n_region = len(regions)

# Budget per 10k population
cpp_df = pd.read_csv('../../data/hseBudget_scaledPer10k.csv')
cost = cpp_df.set_index('Region').loc[regions, 'CostPerPerson'].values

print(f'Samples: {len(raw_df)}')
print(f'Regions: {n_region}')
pd.DataFrame({'Region': regions, 'BudgetPer10k': cost.round(2)})

Samples: 80000
Regions: 6


,Region,BudgetPer10k
0,HSE Dublin and Midlands,3.41
1,HSE Dublin and North East,2.86
2,HSE Dublin and South East,2.74
3,HSE Mid West,2.86
4,HSE South West,2.70
5,HSE West and North West,3.31


## Extract Posterior Samples

In [4]:
# alpha[i] samples: n_samples x n_region
alpha_cols = [f'alpha[{i+1}]' for i in range(n_region)]
alpha_samples = raw_df[alpha_cols].values  # (n_samples, 6)

# mu_lambda samples: n_samples x 1
mu_lambda_samples = raw_df['mu_lambda'].values  # (n_samples,)

print(f'alpha shape: {alpha_samples.shape}')
print(f'mu_lambda shape: {mu_lambda_samples.shape}')
print(f'mu_lambda posterior mean: {mu_lambda_samples.mean():.6f}')
print(f'mu_lambda 95% CI: [{np.percentile(mu_lambda_samples, 2.5):.6f}, {np.percentile(mu_lambda_samples, 97.5):.6f}]')

alpha shape: (80000, 6)
mu_lambda shape: (80000,)
mu_lambda posterior mean: 1.579590
mu_lambda 95% CI: [0.838327, 2.345173]


## Decompose Alpha

For each posterior draw $s$:
- Cost-predicted: $\hat{\alpha}_i^s = \mu_\lambda^s \cdot c_i$
- Residual: $\epsilon_i^s = \alpha_i^s - \hat{\alpha}_i^s$
- Ratio: $\rho_i^s = \hat{\alpha}_i^s \;/\; \alpha_i^s$

In [5]:
# Cost-predicted component: mu_lambda[s] * cost[i]
# Broadcasting: (n_samples, 1) * (1, 6) -> (n_samples, 6)
cost_predicted = mu_lambda_samples[:, None] * cost[None, :]

# Residual
residual = alpha_samples - cost_predicted

# Ratio (cost contribution as fraction of alpha)
ratio = cost_predicted / alpha_samples

In [6]:
summary = pd.DataFrame({
    'Region': regions,
    'BudgetPer10k': cost.round(2),
    'alpha_mean': alpha_samples.mean(axis=0).round(4),
    'cost_pred_mean': cost_predicted.mean(axis=0).round(4),
    'residual_mean': residual.mean(axis=0).round(4),
    'ratio_mean': ratio.mean(axis=0).round(4),
    'ratio_2.5%': np.percentile(ratio, 2.5, axis=0).round(4),
    'ratio_97.5%': np.percentile(ratio, 97.5, axis=0).round(4),
})
summary

,Region,BudgetPer10k,alpha_mean,cost_pred_mean,residual_mean,ratio_mean,ratio_2.5%,ratio_97.5%
0,HSE Dublin and Midlands,3.41,4.0033,5.3794,-1.3762,1.3463,0.7109,2.0159
1,HSE Dublin and North East,2.86,2.2986,4.5176,-2.2190,1.9713,1.0398,2.9707
2,HSE Dublin and South East,2.74,3.0003,4.3252,-1.3249,1.4448,0.7607,2.1657
3,HSE Mid West,2.86,8.1740,4.5201,3.6539,0.5536,0.2923,0.8218
4,HSE South West,2.70,4.4207,4.2699,0.1508,0.9684,0.5101,1.4532
5,HSE West and North West,3.31,6.3446,5.2234,1.1213,0.8244,0.4348,1.2274


## Visualise Decomposition

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(n_region)
short_names = [r.replace('HSE ', '') for r in regions]

cost_means = cost_predicted.mean(axis=0)
resid_means = residual.mean(axis=0)

ax.bar(x, cost_means, label='Cost-predicted', color='steelblue')
ax.bar(x, resid_means, bottom=cost_means, label='Residual', color='coral')

ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=30, ha='right')
ax.set_ylabel('Alpha (baseline trolley rate per 10k)')
ax.set_title('V5: Alpha Decomposition — Budget Contribution vs Residual')
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
axes = axes.flatten()

for i, (ax, region) in enumerate(zip(axes, regions)):
    short = region.replace('HSE ', '')
    ax.hist(ratio[:, i], bins=60, density=True, alpha=0.7, color='steelblue')
    ax.axvline(ratio[:, i].mean(), color='red', linestyle='--', label=f'mean={ratio[:, i].mean():.2f}')
    ax.axvline(1.0, color='black', linestyle=':', alpha=0.5, label='ratio=1')
    ax.set_title(short)
    ax.legend(fontsize=8)
    ax.set_xlabel('Cost / Alpha ratio')

fig.suptitle('Posterior Distribution of Cost Contribution Ratio per Region', fontsize=13)
plt.tight_layout()
plt.show()

## mu_lambda Posterior

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mu_lambda_samples, bins=80, density=True, alpha=0.7, color='steelblue')
ax.axvline(0, color='black', linestyle=':', alpha=0.5)
ax.axvline(mu_lambda_samples.mean(), color='red', linestyle='--',
           label=f'mean={mu_lambda_samples.mean():.6f}')

ci = np.percentile(mu_lambda_samples, [2.5, 97.5])
ax.axvspan(ci[0], ci[1], alpha=0.15, color='red', label=f'95% CI: [{ci[0]:.6f}, {ci[1]:.6f}]')

ax.set_xlabel('mu_lambda')
ax.set_title('Posterior of mu_lambda (cost effect on baseline)')
ax.legend()
plt.tight_layout()
plt.show()

prob_positive = (mu_lambda_samples > 0).mean()
print(f'P(mu_lambda > 0) = {prob_positive:.4f}')

## Budget vs Alpha Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

alpha_means = alpha_samples.mean(axis=0)
alpha_lower = np.percentile(alpha_samples, 2.5, axis=0)
alpha_upper = np.percentile(alpha_samples, 97.5, axis=0)

ax.errorbar(cost, alpha_means, 
            yerr=[alpha_means - alpha_lower, alpha_upper - alpha_means],
            fmt='o', capsize=5, color='steelblue', markersize=8)

# Regression line from mu_lambda posterior mean
c_range = np.linspace(cost.min() * 0.95, cost.max() * 1.05, 100)
ax.plot(c_range, mu_lambda_samples.mean() * c_range, 
        color='red', linestyle='--', label=f'mu_lambda * cost')

for i, r in enumerate(regions):
    ax.annotate(r.replace('HSE ', ''), (cost[i], alpha_means[i]),
                textcoords='offset points', xytext=(8, 5), fontsize=8)

ax.set_xlabel('HR Budget per 10k population')
ax.set_ylabel('Alpha (baseline trolley rate per 10k)')
ax.set_title('V5: Budget vs Baseline with Hierarchical Fit')
ax.legend()
plt.tight_layout()
plt.show()